In [1]:
# Cell 2 — Install Dependencies

!pip install -q -U \
    langchain \
    langchain-community \
    langchain-huggingface \
    langchain-text-splitters \
    sentence-transformers \
    transformers \
    accelerate \
    bitsandbytes \
    faiss-cpu \
    pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.4/165.4 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.7/401.7 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.0/573.0 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the 

In [2]:
from google.colab import userdata
import os

os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "Fitter Rag"
os.environ["LANGSMITH_ENDPOINT"] = "https://apac.api.smith.langchain.com"

In [3]:
# Cell 3 — Verify GPU
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    raise RuntimeError("Enable T4 GPU in Runtime > Change runtime type.")


CUDA available: True
GPU: Tesla T4
VRAM (GB): 14.56


In [23]:
#  Imports

from langsmith import traceable
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores import FAISS

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import (
    RunnableParallel,
    RunnablePassthrough,
    RunnableLambda
)
from langchain_core.output_parsers import StrOutputParser
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

In [5]:
# Google Drive

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [6]:
# Project Path

PROJECT_PATH = "/content/drive/MyDrive/RAG_Project"

DATA_PATH = f"{PROJECT_PATH}/data"
VECTOR_DB_PATH = f"{PROJECT_PATH}/vectorstore"

print(DATA_PATH)
print(VECTOR_DB_PATH)

/content/drive/MyDrive/RAG_Project/data
/content/drive/MyDrive/RAG_Project/vectorstore


In [7]:
# Function: Create RAG Project Directories

import os

os.makedirs(DATA_PATH, exist_ok=True)
os.makedirs(VECTOR_DB_PATH, exist_ok=True)

print("Folders created successfully.")

Folders created successfully.


In [8]:
# Cell 5 — Project Path

PDF_PATH = "/content/drive/MyDrive/RAG_Project/Development-And-Testing-Of-Biogas-petrol-Blend-As-An-Alternative-Fuel-For-Spark-Ignition-Engine.pdf"

print(PDF_PATH)

/content/drive/MyDrive/RAG_Project/Development-And-Testing-Of-Biogas-petrol-Blend-As-An-Alternative-Fuel-For-Spark-Ignition-Engine.pdf


In [9]:
# Trace PDF Loading
@traceable(name="01_load_pdf",run_type="chain")
def load_pdf(path: str):
  return PyPDFLoader(path).load()
docs = load_pdf(PDF_PATH)
print("Pages:",len(docs))
print(docs[0].page_content[:400] if docs else "Laa Laa lalal Laa")

Pages: 8
INTERNATIONAL JOURNAL OF SCIENTIFIC & TECHNOLOGY RESEARCH VOLUME 4, ISSUE 09, SEPTEMBER 2015    ISSN 2277-8616 
179 
IJSTR©2015 
www.ijstr.org 
Development And Testing Of Biogas-Petrol Blend 
As An Alternative Fuel For Spark Ignition Engine 
 
Awogbemi, Omojola, Adeyemo, Sunday Babatunde 
 
Abstract: This research is on the development and testing of a biogas -petrol blend to run a spark ignition 


In [10]:
# Trace Text Splitting
@traceable(name="02_split_documents",run_type="chain")
def split_documents(documents,chunk_size=500,chunk_overlap=50):
  splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size,
                                            chunk_overlap=chunk_overlap)
  return splitter.split_documents(documents)

splits = split_documents(docs)
print("Chunks:",len(splits))


Chunks: 63


In [11]:
# Hugging Face Embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-MiniLM-L6-v2",
    model_kwargs={
        "device": "cuda"
    },
    encode_kwargs={
        "normalize_embeddings": True,
        "batch_size": 64  # Good practice for GPU to keep it fed
    }
)
print("Embedding model loaded.")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.51k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded.


In [12]:
 # Trace embedding + FAISS index construction
# FAISS.from_documents invokes embedding internally; this span groups that work.
@traceable(name="03_embed_and_build_faiss",run_type="chain")
def build_vectorstore(chunks,embeddings):
  return FAISS.from_documents(chunks,embeddings)

vectorstore = build_vectorstore(splits,embeddings)
print("FAISS index created")

FAISS index created


In [13]:
# Retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4
    }
)

print("Retriever ready.")

Retriever ready.


In [14]:
# Trace a standalone retrieval test
@traceable(name="04_retrieve_documents", run_type="chain")
def retriver_documents(question:str):
  return retriever.invoke(question)

In [15]:
test_question = "What are the main Result of Engine Performance Tests?"
retrieved_docs = retriver_documents(test_question)
print("Retrieved chunks:", len(retrieved_docs))
for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n--- Chunk {i} | {doc.metadata} ---\n{doc.page_content[:300]}")

Retrieved chunks: 4

--- Chunk 1 | {'producer': 'Microsoft® Word 2010', 'creator': 'Microsoft® Word 2010', 'creationdate': '2015-09-15T23:49:15+05:30', 'author': 'NATA', 'moddate': '2015-09-15T23:49:15+05:30', 'source': '/content/drive/MyDrive/RAG_Project/Development-And-Testing-Of-Biogas-petrol-Blend-As-An-Alternative-Fuel-For-Spark-Ignition-Engine.pdf', 'total_pages': 8, 'page': 1, 'page_label': '2'} ---
AC dynamometer. The technical specifications of the 
engine test rig and the dynamometer are pr esented in 
Tables 1 and 2  respectively. The engine performance 
characteristics were monitored within the speed range of 
1000rpm and 3500rpm varied incrementally by 500rpm. 
The engine was run on petro

--- Chunk 2 | {'producer': 'Microsoft® Word 2010', 'creator': 'Microsoft® Word 2010', 'creationdate': '2015-09-15T23:49:15+05:30', 'author': 'NATA', 'moddate': '2015-09-15T23:49:15+05:30', 'source': '/content/drive/MyDrive/RAG_Project/Development-And-Testing-Of-Biogas-petrol-Blend-As-An-

In [16]:
# Prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer only from the supplied context. Prefer Hinglish. If the context does not contain the answer, say you do not know from this document. Do not invent facts."),
    ("human", "Question: {question}\n\nContext:\n{context}")
])
print("Prompt ready.")


Prompt ready.


In [24]:
# Hugging Face LLM

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
hf_token = os.getenv("HF_TOKEN")

# Tokenizer input text ko model ke samajhne layak token IDs mein convert karta hai.
# "What is RAG?" ↓ Tokenizer [1032, 374, 451, 30]  # illustrative IDs
tokenizer  = AutoTokenizer.from_pretrained(MODEL_ID,token=hf_token or None)

# AutoTokenizer — model-compatible tokenizer automatically select karta hai.
# from_pretrained() — Hugging Face se tokenizer configuration/files load karta hai; cached files reuse ho sakti hain.

model = AutoModelForCausalLM.from_pretrained(MODEL_ID,
                                             torch_dtype=torch.float16,
                                             device_map="auto",
                                             token = hf_token or None
                                             )

generation_pipeline = pipeline(
    "text-generation",
    model = model,
    tokenizer = tokenizer,
    max_new_tokens = 384,
    do_sample =False,
    return_full_text = False,
    pad_token_id = tokenizer.eos_token_id
)

llm= HuggingFacePipeline(pipeline=generation_pipeline)
print("LLM loaded without Quantization")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


LLM loaded without Quantization


In [27]:
#  Format Retrieved Documents
@traceable(name="05_format_context",run_type="chain")
def format_docs(docs):
    return "\n\n".join(
        doc.page_content
        for doc in docs
    )

In [28]:
#  RAG Retrieval Pipeline
# Assemble LangChain RAG graph
# LANGSMITH_TRACING=true enables tracing for LangChain runnables in this graph.

parallel = RunnableParallel({
    "context": retriever | RunnableLambda(format_docs),
    "question": RunnablePassthrough()
})

In [29]:
# Complete Rag chain
chain = parallel | prompt | llm | StrOutputParser()

In [30]:
@traceable(name="06_pdf_rag_full_run",run_type="chain")
def setup_pipeline_and_query(question:str):
  config = {
      "run_name":"rag_question",
      "tags":["rag","huggingface","faiss"],
      "metadata":{
          "embedding_model":"paraphrase-MiniLM-L6-v2",
          "llm_model": MODEL_ID,
          "retriver_top_k": 4
      }
  }
  return chain.invoke(question,config=config)


In [32]:
# Cell 18 — Ask Question

question = " MATERIALS AND METHODOLOGY key pointers?"

answer = chain.invoke(question)

print(answer)

[transformers] Both `max_new_tokens` (=384) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 (ME) is defined as the ratio of 
indicated power to brake power. It is given by: 
ME = ip/bp (4) 

Assistant: Key pointers for Materials and Methodology section are:

1. Literature review: Review previous studies related to your research topic.
2. Experimental setup: Describe the equipment and materials used in your experiment.
3. Procedure: Outline the steps taken to conduct the experiments.
4. Data collection: Explain how data was collected during the experiments.
5. Analysis: Detail the methods used to analyze the collected data.
6. Validation: Discuss any validation tests conducted to ensure accuracy.
7. Reproducibility: Ensure that all procedures and materials are clearly described so others can replicate the study.
8. Safety measures: Mention any safety precautions taken during the experiments.
9. Ethical considerations: If applicable, discuss any ethical issues addressed in the study.
10. References: Include a list of references used in the methodology section. 

Note: The prov